# Data Ingestion — Olist → Databricks Delta Lake

**Purpose**: pull the [Olist Brazilian E-Commerce dataset](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce)
from Kaggle and land it in Databricks as managed Delta tables, so every specialist agent's
tools can query it via SQL instead of re-reading CSVs on every call.

**Why Unity Catalog Volumes, not DBFS root**: this workspace has the public DBFS root
disabled (a Unity Catalog security policy), so raw file upload has to go through a managed
Volume instead — `workspace.olist.raw_files`. This is also just the modern, recommended path
regardless of that policy.

**Why a serverless SQL warehouse, not a cluster**: this workspace has no cluster compute
policy configured, so provisioning a general-purpose cluster times out. The serverless SQL
warehouse was already available and is actually a better fit here — it starts in seconds and
is billed per-query, not per-hour.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from databricks.sdk import WorkspaceClient
from backend.config import settings

DATA_DIR = Path.cwd().parent / "data" / "raw"
CATALOG = settings.databricks_catalog
SCHEMA = settings.databricks_schema
VOLUME = "raw_files"

CSV_TABLES = [
    ("olist_customers_dataset.csv",           "customers"),
    ("olist_orders_dataset.csv",              "orders"),
    ("olist_order_items_dataset.csv",         "order_items"),
    ("olist_order_payments_dataset.csv",      "order_payments"),
    ("olist_order_reviews_dataset.csv",       "order_reviews"),
    ("olist_products_dataset.csv",            "products"),
    ("olist_sellers_dataset.csv",             "sellers"),
    ("olist_geolocation_dataset.csv",         "geolocation"),
    ("product_category_name_translation.csv", "product_category_translation"),
]

client = WorkspaceClient(host=settings.databricks_host, token=settings.databricks_token)
print(f"Connected to {settings.databricks_host}")

Connected to https://dbc-8307750a-af67.cloud.databricks.com


## Step 1 — Ensure the catalog schema and managed volume exist

Idempotent: safe to re-run without duplicating anything.

In [2]:
from databricks.sdk.service.catalog import VolumeType

schemas = [s.name for s in client.schemas.list(catalog_name=CATALOG)]
if SCHEMA not in schemas:
    client.schemas.create(name=SCHEMA, catalog_name=CATALOG)
    print(f"Created schema: {SCHEMA}")
else:
    print(f"Schema exists: {SCHEMA}")

volumes = [v.name for v in client.volumes.list(catalog_name=CATALOG, schema_name=SCHEMA)]
if VOLUME not in volumes:
    client.volumes.create(catalog_name=CATALOG, schema_name=SCHEMA, name=VOLUME, volume_type=VolumeType.MANAGED)
    print(f"Created volume: {VOLUME}")
else:
    print(f"Volume exists: {VOLUME}")

Schema exists: olist


Volume exists: raw_files


## Step 2 — Upload the raw CSVs to the volume

These files are gitignored (`data/raw/`) — pulled fresh via the Kaggle API on each machine rather than committed to the repo.

In [3]:
for filename, _ in CSV_TABLES:
    local_path = DATA_DIR / filename
    volume_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{filename}"
    with open(local_path, "rb") as f:
        client.files.upload(volume_path, f, overwrite=True)
    print(f"uploaded {filename}")

uploaded olist_customers_dataset.csv


uploaded olist_orders_dataset.csv


uploaded olist_order_items_dataset.csv


uploaded olist_order_payments_dataset.csv


uploaded olist_order_reviews_dataset.csv


uploaded olist_products_dataset.csv


uploaded olist_sellers_dataset.csv


uploaded olist_geolocation_dataset.csv


uploaded product_category_name_translation.csv


## Step 3 — Create Delta tables via the SQL warehouse

`read_files()` + `CREATE TABLE ... AS SELECT` reads directly from the volume and materializes a managed Delta table — no cluster, no manual schema inference.

In [ ]:
from databricks.sdk.service.sql import StatementState
import time

warehouse_id = settings.databricks_warehouse_id or list(client.warehouses.list())[0].id
print(f"Using warehouse: {warehouse_id}")

def run_sql(stmt: str):
    response = client.statement_execution.execute_statement(statement=stmt, warehouse_id=warehouse_id, wait_timeout="50s")
    while response.status and response.status.state in (StatementState.PENDING, StatementState.RUNNING):
        time.sleep(2)
        response = client.statement_execution.get_statement(response.statement_id)
    if response.status and response.status.state != StatementState.SUCCEEDED:
        error = response.status.error.message if response.status.error else "unknown"
        raise RuntimeError(f"Failed: {error}\nSQL: {stmt}")
    return response

run_sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{SCHEMA}")

# multiLine=true matters specifically for order_reviews: review text can contain embedded
# newlines, and without it the CSV parser splits one logical row into several, corrupting
# downstream columns for ~0.07% of rows (caught and worked around in the Sentiment Agent's
# tools, but fixing it here means new environments don't inherit the bug at all).
for filename, table_name in CSV_TABLES:
    vol_path = f"/Volumes/{CATALOG}/{SCHEMA}/{VOLUME}/{filename}"
    run_sql(f"""
        CREATE OR REPLACE TABLE {CATALOG}.{SCHEMA}.{table_name}
        USING DELTA
        AS SELECT * FROM read_files('{vol_path}', format => 'csv', header => true, inferSchema => true, multiLine => true)
    """)
    print(f"created table: {table_name}")

print("\nAll 9 Delta tables live at workspace.olist.*")

## Result

| Table | Used by |
|---|---|
| `orders` | Finance, Sales, Operations, Risk |
| `order_items` | Finance, Sales, Growth |
| `order_payments` | Finance, Risk |
| `order_reviews` | Sentiment (RAG, pending) |
| `customers` | Sales, Growth, Risk |
| `sellers` | Operations, Risk, Compliance |
| `products` | Sales, Growth |
| `geolocation` | Growth, Operations |
| `product_category_translation` | Sales, Growth |